# Entity Extraction Evaluation — First 100 Rows

This notebook evaluates the first **100 rows only** from:

- `ground_truth_entities_100.csv` — ground truth
- `llm_entities_vs_gt_100.csv` — LLM predictions

It evaluates two features:

- `canonical_name`
- `entity_type`

Only these metrics are calculated:

- Precision
- Recall
- F1-score
- BERTScore F1

## Metric interpretation

### `canonical_name`

Precision, recall, and F1-score are calculated from token overlap between the normalized reference and predicted entity names. BERTScore F1 measures semantic similarity.

### `entity_type`

Precision, recall, and F1-score are macro-averaged across:

- `DiseaseCondition`
- `Treatment`
- `Symptom`
- `Test`

BERTScore F1 is also calculated over the entity-type label strings because it was requested, although macro Precision, Recall, and F1-score are the main metrics for this categorical feature.

## 1. Install dependencies

In [ ]:
%pip install -q pandas numpy scikit-learn bert-score transformers torch

## 2. Imports and configuration

In [ ]:
from pathlib import Path
from collections import Counter
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import torch

from bert_score import score as bertscore
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 30)

GROUND_TRUTH_FILENAME = "ground_truth_entities_100.csv"
LLM_FILENAME = "llm_entities_vs_gt_100.csv"

NUMBER_OF_ROWS = 100
ENTITY_TYPES = [
    "DiseaseCondition",
    "Treatment",
    "Symptom",
    "Test",
]

BERTSCORE_MODEL = "xlm-roberta-base"
BERTSCORE_BATCH_SIZE = 32

OUTPUT_DIR = Path("entity_evaluation_first_100_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load and align the first 100 rows

In [ ]:
def resolve_input_file(filename: str) -> Path:
    """Find an input file in common notebook locations."""
    candidate_paths = [
        Path(filename),
        Path("/mnt/data") / filename,
        Path("/content") / filename,
        Path("/kaggle/working") / filename,
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Could not find {filename!r}. Upload it beside the notebook "
        "or update the filename in the configuration cell."
    )


ground_truth_path = resolve_input_file(GROUND_TRUTH_FILENAME)
llm_path = resolve_input_file(LLM_FILENAME)

ground_truth = pd.read_csv(ground_truth_path).head(NUMBER_OF_ROWS).copy()
llm_output = pd.read_csv(llm_path).head(NUMBER_OF_ROWS).copy()

required_columns = {
    "question",
    "answer",
    "entity_type",
    "canonical_name",
}

missing_gt = required_columns - set(ground_truth.columns)
missing_llm = required_columns - set(llm_output.columns)

if missing_gt:
    raise ValueError(
        f"Ground-truth file is missing columns: {sorted(missing_gt)}"
    )

if missing_llm:
    raise ValueError(
        f"LLM file is missing columns: {sorted(missing_llm)}"
    )

if len(ground_truth) < NUMBER_OF_ROWS:
    raise ValueError(
        f"Ground-truth file contains only {len(ground_truth)} rows."
    )

if len(llm_output) < NUMBER_OF_ROWS:
    raise ValueError(
        f"LLM file contains only {len(llm_output)} rows."
    )

for frame in (ground_truth, llm_output):
    for column in required_columns:
        frame[column] = (
            frame[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

if not ground_truth["question"].equals(llm_output["question"]):
    raise ValueError(
        "The first 100 rows are not aligned by question."
    )

if not ground_truth["answer"].equals(llm_output["answer"]):
    raise ValueError(
        "The first 100 rows are not aligned by answer."
    )

comparison = pd.DataFrame(
    {
        "row_id": np.arange(1, NUMBER_OF_ROWS + 1),
        "question": ground_truth["question"],
        "answer": ground_truth["answer"],
        "gt_canonical_name": ground_truth["canonical_name"],
        "pred_canonical_name": llm_output["canonical_name"],
        "gt_entity_type": ground_truth["entity_type"],
        "pred_entity_type": llm_output["entity_type"],
    }
)

print("Rows used:", len(comparison))
print("Question and answer alignment: confirmed")
display(comparison.head())

## 4. Arabic normalization

In [ ]:
ARABIC_DIACRITICS_RE = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
)
NON_WORD_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)
WHITESPACE_RE = re.compile(r"\s+")

ARABIC_CHARACTER_MAP = str.maketrans(
    {
        "أ": "ا",
        "إ": "ا",
        "آ": "ا",
        "ٱ": "ا",
        "ى": "ي",
        "ؤ": "و",
        "ئ": "ي",
    }
)


def normalize_arabic(text: str) -> str:
    """Apply conservative Arabic normalization."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("ـ", "")
    text = ARABIC_DIACRITICS_RE.sub("", text)
    text = text.translate(ARABIC_CHARACTER_MAP)
    text = NON_WORD_RE.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text).strip().lower()
    return text


def tokenize(text: str) -> list[str]:
    return [
        token
        for token in normalize_arabic(text).split()
        if token
    ]


def safe_divide(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


comparison["gt_canonical_name_normalized"] = (
    comparison["gt_canonical_name"].map(normalize_arabic)
)
comparison["pred_canonical_name_normalized"] = (
    comparison["pred_canonical_name"].map(normalize_arabic)
)

## 5. Canonical-name Precision, Recall, and F1-score

In [ ]:
def canonical_name_token_metrics(
    reference: str,
    prediction: str,
) -> dict:
    """Calculate token-overlap precision, recall, and F1."""
    reference_tokens = Counter(tokenize(reference))
    prediction_tokens = Counter(tokenize(prediction))

    overlap = sum(
        (reference_tokens & prediction_tokens).values()
    )

    reference_count = sum(reference_tokens.values())
    prediction_count = sum(prediction_tokens.values())

    precision = safe_divide(overlap, prediction_count)
    recall = safe_divide(overlap, reference_count)
    f1 = safe_divide(
        2 * precision * recall,
        precision + recall,
    )

    return {
        "canonical_name_precision": precision,
        "canonical_name_recall": recall,
        "canonical_name_f1": f1,
    }


canonical_token_scores = comparison.apply(
    lambda row: canonical_name_token_metrics(
        row["gt_canonical_name_normalized"],
        row["pred_canonical_name_normalized"],
    ),
    axis=1,
    result_type="expand",
)

comparison = pd.concat(
    [comparison, canonical_token_scores],
    axis=1,
)

canonical_name_precision = (
    comparison["canonical_name_precision"].mean()
)
canonical_name_recall = (
    comparison["canonical_name_recall"].mean()
)
canonical_name_f1 = (
    comparison["canonical_name_f1"].mean()
)

print(
    f"Canonical-name Precision: {canonical_name_precision:.4f}"
)
print(
    f"Canonical-name Recall:    {canonical_name_recall:.4f}"
)
print(
    f"Canonical-name F1-score:  {canonical_name_f1:.4f}"
)

## 6. Canonical-name BERTScore F1

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("BERTScore device:", device)

_, _, canonical_name_bertscore_f1_values = bertscore(
    cands=comparison[
        "pred_canonical_name_normalized"
    ].tolist(),
    refs=comparison[
        "gt_canonical_name_normalized"
    ].tolist(),
    model_type=BERTSCORE_MODEL,
    batch_size=BERTSCORE_BATCH_SIZE,
    device=device,
    verbose=True,
    rescale_with_baseline=False,
)

comparison["canonical_name_bertscore_f1"] = (
    canonical_name_bertscore_f1_values.cpu().numpy()
)

canonical_name_bertscore_f1 = (
    comparison["canonical_name_bertscore_f1"].mean()
)

print(
    "Canonical-name BERTScore F1: "
    f"{canonical_name_bertscore_f1:.4f}"
)

## 7. Entity-type Precision, Recall, and F1-score

In [ ]:
entity_type_precision = precision_score(
    comparison["gt_entity_type"],
    comparison["pred_entity_type"],
    labels=ENTITY_TYPES,
    average="macro",
    zero_division=0,
)

entity_type_recall = recall_score(
    comparison["gt_entity_type"],
    comparison["pred_entity_type"],
    labels=ENTITY_TYPES,
    average="macro",
    zero_division=0,
)

entity_type_f1 = f1_score(
    comparison["gt_entity_type"],
    comparison["pred_entity_type"],
    labels=ENTITY_TYPES,
    average="macro",
    zero_division=0,
)

print(
    f"Entity-type macro Precision: {entity_type_precision:.4f}"
)
print(
    f"Entity-type macro Recall:    {entity_type_recall:.4f}"
)
print(
    f"Entity-type macro F1-score:  {entity_type_f1:.4f}"
)

entity_type_report = pd.DataFrame(
    classification_report(
        comparison["gt_entity_type"],
        comparison["pred_entity_type"],
        labels=ENTITY_TYPES,
        output_dict=True,
        zero_division=0,
    )
).T

display(
    entity_type_report.loc[
        ENTITY_TYPES,
        ["precision", "recall", "f1-score", "support"],
    ].style.format(
        {
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "f1-score": "{:.4f}",
            "support": "{:.0f}",
        }
    )
)

## 8. Entity-type BERTScore F1

In [ ]:
_, _, entity_type_bertscore_f1_values = bertscore(
    cands=comparison["pred_entity_type"].tolist(),
    refs=comparison["gt_entity_type"].tolist(),
    model_type=BERTSCORE_MODEL,
    batch_size=BERTSCORE_BATCH_SIZE,
    device=device,
    verbose=True,
    rescale_with_baseline=False,
)

comparison["entity_type_bertscore_f1"] = (
    entity_type_bertscore_f1_values.cpu().numpy()
)

entity_type_bertscore_f1 = (
    comparison["entity_type_bertscore_f1"].mean()
)

print(
    "Entity-type BERTScore F1: "
    f"{entity_type_bertscore_f1:.4f}"
)

## 9. Final results table

In [ ]:
results = pd.DataFrame(
    [
        {
            "Feature": "canonical_name",
            "Precision": canonical_name_precision,
            "Recall": canonical_name_recall,
            "F1-score": canonical_name_f1,
            "BERTScore F1": canonical_name_bertscore_f1,
        },
        {
            "Feature": "entity_type",
            "Precision": entity_type_precision,
            "Recall": entity_type_recall,
            "F1-score": entity_type_f1,
            "BERTScore F1": entity_type_bertscore_f1,
        },
    ]
)

display(
    results.style.format(
        {
            "Precision": "{:.4f}",
            "Recall": "{:.4f}",
            "F1-score": "{:.4f}",
            "BERTScore F1": "{:.4f}",
        }
    )
)

## 10. Export results

In [ ]:
summary_output_path = (
    OUTPUT_DIR / "first_100_metric_summary.csv"
)
detailed_output_path = (
    OUTPUT_DIR / "first_100_row_comparison.csv"
)
entity_type_report_output_path = (
    OUTPUT_DIR / "first_100_entity_type_report.csv"
)

results.to_csv(
    summary_output_path,
    index=False,
    encoding="utf-8-sig",
)

comparison.to_csv(
    detailed_output_path,
    index=False,
    encoding="utf-8-sig",
)

entity_type_report.to_csv(
    entity_type_report_output_path,
    encoding="utf-8-sig",
)

print("Saved:")
print(" -", summary_output_path)
print(" -", detailed_output_path)
print(" -", entity_type_report_output_path)